In [1]:
import torch
import torch.nn as nn
from torchvision import models
from PIL import Image
from torch.utils.data import Dataset

transform = (
    models.EfficientNet_B0_Weights
    .DEFAULT
    .transforms()
)

In [2]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

UTA_ROOT = (
    PROJECT_ROOT
    / "processed"
    / "UTA"
    / "sequences"
)

rows = []

for subject_dir in sorted(UTA_ROOT.iterdir()):

    subject = subject_dir.name

    for cls in ["0", "5", "10"]:

        frame_dir = subject_dir / cls

        for img_path in frame_dir.glob("*.jpg"):

            rows.append({
                "path": str(img_path),
                "subject": subject,
                "label": int(cls)
            })

df = pd.DataFrame(rows)

print(df.head())
print(df.shape)

                                                path subject  label
0  c:\Users\clark\OneDrive\Desktop\drowsiness_pro...      01      0
1  c:\Users\clark\OneDrive\Desktop\drowsiness_pro...      01      0
2  c:\Users\clark\OneDrive\Desktop\drowsiness_pro...      01      0
3  c:\Users\clark\OneDrive\Desktop\drowsiness_pro...      01      0
4  c:\Users\clark\OneDrive\Desktop\drowsiness_pro...      01      0
(87679, 3)


In [3]:
label_map = {
    0: 0,
    5: 1,
    10: 2
}

df["label"] = df["label"].map(label_map)

In [4]:
TEST_SUBJECTS = [
    "01","06","11","16","21",
    "26","31","36","41","46"
]

In [5]:
df = df[
    ~df["subject"].isin(TEST_SUBJECTS)
]

In [6]:
import random

random.seed(42)

subjects = sorted(
    df["subject"].unique()
)

val_subjects = random.sample(
    list(subjects),
    8
)

train_subjects = [
    s for s in subjects
    if s not in val_subjects
]

print("Train:", len(train_subjects))
print("Val:", len(val_subjects))

Train: 30
Val: 8


In [7]:
train_df = df[
    df["subject"].isin(train_subjects)
]

val_df = df[
    df["subject"].isin(val_subjects)
]

print(train_df["label"].value_counts())
print(val_df["label"].value_counts())

label
0    19124
1    18336
2    17969
Name: count, dtype: int64
label
2    4954
1    4856
0    4838
Name: count, dtype: int64


In [8]:
def sample_subject(df_sub):

    sampled = []

    for s in df_sub["subject"].unique():

        sub = df_sub[
            df_sub["subject"] == s
        ]

        for c in [0,1,2]:

            cls = sub[
                sub["label"] == c
            ]

            n = min(
                200,
                len(cls)
            )

            sampled.append(
                cls.sample(
                    n=n,
                    random_state=42
                )
            )

    return pd.concat(sampled)

train_df = sample_subject(train_df)
val_df = sample_subject(val_df)

print("Sampled train size:", len(train_df))
print("Sampled validation size:", len(val_df))

print("\nSampled train class counts:")
print(train_df["label"].value_counts().sort_index())

print("\nSampled validation class counts:")
print(val_df["label"].value_counts().sort_index())

Sampled train size: 17726
Sampled validation size: 4800

Sampled train class counts:
label
0    6000
1    5926
2    5800
Name: count, dtype: int64

Sampled validation class counts:
label
0    1600
1    1600
2    1600
Name: count, dtype: int64


In [9]:
from PIL import Image
from torch.utils.data import Dataset

transform = (
    models.EfficientNet_B0_Weights
    .DEFAULT
    .transforms()
)

class UTAFrameDataset(Dataset):

    def __init__(
        self,
        df,
        transform
    ):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        img = Image.open(
            row["path"]
        ).convert("RGB")

        img = self.transform(img)

        label = int(
            row["label"]
        )

        return img, label

In [10]:
train_dataset = UTAFrameDataset(
    train_df,
    transform
)

val_dataset = UTAFrameDataset(
    val_df,
    transform
)

print(len(train_dataset))
print(len(val_dataset))

17726
4800


In [11]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

In [12]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = models.efficientnet_b0(
    weights=None
)

model.classifier = nn.Sequential(
    nn.Dropout(0.3),
    nn.Linear(1280, 512),
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(512, 2)
)

model.load_state_dict(
    torch.load(
        PROJECT_ROOT
        / "models"
        / "checkpoints"
        / "best_cnn.pth",
        map_location=device
    )
)

<All keys matched successfully>

In [13]:
model.classifier[-1] = nn.Linear(
    512,
    3
)

In [23]:
for param in model.parameters():
    param.requires_grad = False

for param in model.features[-4:].parameters():
    param.requires_grad = True

for param in model.classifier.parameters():
    param.requires_grad = True

In [24]:
sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

4356299

In [25]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad,
           model.parameters()),
    lr=5e-5,
    weight_decay=1e-5
)


In [26]:
sum(p.numel() for p in model.parameters() if p.requires_grad)
print(model.classifier)


Sequential(
  (0): Dropout(p=0.3, inplace=False)
  (1): Linear(in_features=1280, out_features=512, bias=True)
  (2): ReLU()
  (3): Dropout(p=0.3, inplace=False)
  (4): Linear(in_features=512, out_features=3, bias=True)
)


In [27]:
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report
)
from tqdm import tqdm
import copy

In [28]:
def train_one_epoch(model, loader):

    model.train()

    running_loss = 0
    preds = []
    labels = []

    for x, y in tqdm(loader):

        x = x.to(device)
        y = y.to(device)

        optimizer.zero_grad()

        out = model(x)

        loss = criterion(out, y)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        pred = out.argmax(1)

        preds.extend(pred.cpu().numpy())
        labels.extend(y.cpu().numpy())

    loss = running_loss / len(loader)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(
        labels,
        preds,
        average="macro"
    )

    return loss, acc, f1

In [29]:
def validate(model, loader):

    model.eval()

    running_loss = 0
    preds = []
    labels = []

    with torch.no_grad():

        for x, y in tqdm(loader):

            x = x.to(device)
            y = y.to(device)

            out = model(x)

            loss = criterion(out, y)

            running_loss += loss.item()

            pred = out.argmax(1)

            preds.extend(pred.cpu().numpy())
            labels.extend(y.cpu().numpy())

    loss = running_loss / len(loader)

    acc = accuracy_score(labels, preds)

    f1 = f1_score(
        labels,
        preds,
        average="macro"
    )

    report = classification_report(
        labels,
        preds,
        target_names=[
            "Alert",
            "Low Vigilant",
            "Drowsy"
        ],
        digits=4
    )

    return loss, acc, f1, report

In [30]:
x, y = next(iter(train_loader))
print(x.shape)
print(y.shape)

torch.Size([32, 3, 224, 224])
torch.Size([32])


In [31]:
NUM_EPOCHS = 20
PATIENCE = 5

best_f1 = 0
counter = 0

best_model = None
for epoch in range(NUM_EPOCHS):

    print(f"\nEpoch {epoch+1}/{NUM_EPOCHS}")

    train_loss, train_acc, train_f1 = train_one_epoch(
        model,
        train_loader
    )

    val_loss, val_acc, val_f1, report = validate(
        model,
        val_loader
    )

    print(
        f"Train Loss: {train_loss:.4f}"
        f" | Train Acc: {train_acc:.4f}"
        f" | Train F1: {train_f1:.4f}"
    )

    print(
        f"Val Loss: {val_loss:.4f}"
        f" | Val Acc: {val_acc:.4f}"
        f" | Val F1: {val_f1:.4f}"
    )

    print(report)

    if val_f1 > best_f1:

        best_f1 = val_f1
        counter = 0

        best_model = copy.deepcopy(
            model.state_dict()
        )

        torch.save(
            best_model,
            PROJECT_ROOT
            / "models"
            / "checkpoints"
            / "best_cnn_3class.pth"
        )

        print("✅ Best model saved.")

    else:

        counter += 1
        print(f"No improvement ({counter}/{PATIENCE})")

        if counter >= PATIENCE:
            print("Early stopping.")
            break


Epoch 1/20


100%|██████████| 150/150 [00:18<00:00,  8.29it/s]


Train Loss: 1.0117 | Train Acc: 0.5042 | Train F1: 0.4998
Val Loss: 1.2849 | Val Acc: 0.3354 | Val F1: 0.3276
              precision    recall  f1-score   support

       Alert     0.3314    0.4188    0.3700      1600
Low Vigilant     0.3196    0.3875    0.3503      1600
      Drowsy     0.3819    0.2000    0.2625      1600

    accuracy                         0.3354      4800
   macro avg     0.3443    0.3354    0.3276      4800
weighted avg     0.3443    0.3354    0.3276      4800

✅ Best model saved.

Epoch 2/20


100%|██████████| 150/150 [00:18<00:00,  8.24it/s]


Train Loss: 0.6764 | Train Acc: 0.7186 | Train F1: 0.7161
Val Loss: 1.6743 | Val Acc: 0.3398 | Val F1: 0.3363
              precision    recall  f1-score   support

       Alert     0.3473    0.4363    0.3867      1600
Low Vigilant     0.3084    0.3219    0.3150      1600
      Drowsy     0.3732    0.2612    0.3074      1600

    accuracy                         0.3398      4800
   macro avg     0.3430    0.3398    0.3363      4800
weighted avg     0.3430    0.3398    0.3363      4800

✅ Best model saved.

Epoch 3/20


...

Train Loss: 0.0221 | Train Acc: 0.9924 | Train F1: 0.9924
Val Loss: 3.3259 | Val Acc: 0.3790 | Val F1: 0.3759
              precision    recall  f1-score   support

       Alert     0.3866    0.5081    0.4391      1600
Low Vigilant     0.2972    0.2806    0.2887      1600
      Drowsy     0.4696    0.3481    0.3999      1600

    accuracy                         0.3790      4800
   macro avg     0.3845    0.3790    0.3759      4800
weighted avg     0.3845    0.3790    0.3759      4800

No improvement (3/5)

Epoch 17/20


100%|██████████| 150/150 [00:40<00:00,  3.71it/s]


Train Loss: 0.0175 | Train Acc: 0.9939 | Train F1: 0.9939
Val Loss: 3.3002 | Val Acc: 0.3944 | Val F1: 0.3881
              precision    recall  f1-score   support

       Alert     0.4005    0.5256    0.4546      1600
Low Vigilant     0.3143    0.2512    0.2793      1600
      Drowsy     0.4574    0.4062    0.4303      1600

    accuracy                         0.3944      4800
   macro avg     0.3907    0.3944    0.3881      4800
weighted avg     0.3907    0.3944    0.3881      4800

No improvement (4/5)

Epoch 18/20


100%|██████████| 150/150 [00:21<00:00,  6.85it/s]

Train Loss: 0.0160 | Train Acc: 0.9943 | Train F1: 0.9943
Val Loss: 3.3491 | Val Acc: 0.3881 | Val F1: 0.3832
              precision    recall  f1-score   support

       Alert     0.3968    0.5262    0.4524      1600
Low Vigilant     0.3107    0.2712    0.2896      1600
      Drowsy     0.4582    0.3669    0.4075      1600

    accuracy                         0.3881      4800
   macro avg     0.3886    0.3881    0.3832      4800
weighted avg     0.3886    0.3881    0.3832      4800

No improvement (5/5)
Early stopping.


In [22]:
for name, param in model.named_parameters():
    if param.requires_grad:
        print(name)

features.7.0.block.0.0.weight
features.7.0.block.0.1.weight
features.7.0.block.0.1.bias
features.7.0.block.1.0.weight
features.7.0.block.1.1.weight
features.7.0.block.1.1.bias
features.7.0.block.2.fc1.weight
features.7.0.block.2.fc1.bias
features.7.0.block.2.fc2.weight
features.7.0.block.2.fc2.bias
features.7.0.block.3.0.weight
features.7.0.block.3.1.weight
features.7.0.block.3.1.bias
features.8.0.weight
features.8.1.weight
features.8.1.bias
classifier.1.weight
classifier.1.bias
classifier.4.weight
classifier.4.bias
